# Auto-Labeler Training on Colab (A100)

1D U-Net for offline tissue segmentation — trains on 85 labeled studies.

**Setup:** Data is on Google Drive at `G:\My Drive\auto_labeler` (mounted as `/content/drive/MyDrive/auto_labeler`).

In [ ]:
# Mount Google Drive -- yes
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [23]:
!rm -rf /content/drive/MyDrive/auto_labeler/__pycache__
!cd /tmp && rm -rf PyTorch_3 && git clone https://github.com/RonInQu/PyTorch_3.git
!cp /tmp/PyTorch_3/auto_labeler/*.py /content/drive/MyDrive/auto_labeler/
print("Code synced from git")

Cloning into 'PyTorch_3'...
remote: Enumerating objects: 767, done.
remote: Counting objects: 100% (204/204), done.
remote: Compressing objects: 100% (90/90), done.
remote: Total 767 (delta 116), reused 164 (delta 94), pack-reused 563 (from 1)
Receiving objects: 100% (767/767), 336.24 KiB | 25.86 MiB/s, done.
Resolving deltas: 100% (470/470), done.
Code synced from git


In [2]:
# Set working directory to the auto_labeler folder on Google Drive
%cd /content/drive/MyDrive/auto_labeler

/content/drive/MyDrive/auto_labeler


In [5]:
# Verify directory contents
import os
print("Contents of working directory:")
for f in sorted(os.listdir(".")):
    print(f"  {f}")


Contents of working directory:
  __init__.py
  __pycache__
  checkpoints
  config.py
  dataset.py
  evaluate.py
  model.py
  predict.py
  predict_labels
  results
  test_data_8
  train.py
  train_colab.ipynb
  train_colab_v0.ipynb
  training_data


In [3]:
# Verify GPU
import torch
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")

PyTorch: 2.10.0+cu128
CUDA available: True
GPU: NVIDIA A100-SXM4-40GB
VRAM: 42.4 GB


In [ ]:
# Install any missing dependencies
!pip install pandas pyarrow matplotlib -q

In [6]:
# Verify training data exists
import os
data_dir = "training_data"
files = [f for f in os.listdir(data_dir) if f.endswith('.parquet')]
print(f"Training parquets found: {len(files)}")
assert len(files) >= 85, f"Expected 85+ files, got {len(files)}. Check data_dir path."

Training parquets found: 85


In [7]:
# Verify model builds correctly
from model import UNet1D, count_parameters

model = UNet1D(in_channels=1, num_classes=3, base_filters=32, depth=5, kernel_size=7)
print(f"Parameters: {count_parameters(model):,}")

x = torch.randn(2, 1, 4096).cuda()
model = model.cuda()
y = model(x)
print(f"Input: {x.shape} -> Output: {y.shape}")
del model, x, y
torch.cuda.empty_cache()

Parameters: 27,263,875
Input: torch.Size([2, 1, 4096]) -> Output: torch.Size([2, 3, 4096])


## Train

With A100 + batch_size=64, expect 10 minutes for 80 epochs.

In [ ]:
# Run training — ALL parameters come from config.py
%cd /content/drive/MyDrive

!python -m auto_labeler.train \
    --data_dir auto_labeler/training_data \
    --output_dir auto_labeler/checkpoints

# Return to auto_labeler dir
%cd /content/drive/MyDrive/auto_labeler

# Show manifest
!cat checkpoints/manifest.txt

## Evaluate on Test Studies

In [ ]:
# Evaluate on all studies in test_data/ folder
%cd /content/drive/MyDrive

!python -m auto_labeler.evaluate \
    --data_dir auto_labeler/test_data \
    --checkpoint auto_labeler/checkpoints/best_model.pt \
    --output_dir auto_labeler/results \
    --plot

%cd /content/drive/MyDrive/auto_labeler

/content/drive/MyDrive
Device: cuda
Multichannel: False
Evaluating 8 studies...
  33CFB812: F1=0.6359, Acc=0.9025
  Saved: auto_labeler/results/plots/33CFB812_overlay.png
  819421BC: F1=0.7214, Acc=0.9051
  Saved: auto_labeler/results/plots/819421BC_overlay.png
  847A1E3F: F1=0.7041, Acc=0.8985
  Saved: auto_labeler/results/plots/847A1E3F_overlay.png
  8ECEADA6: F1=0.7973, Acc=0.8975
  Saved: auto_labeler/results/plots/8ECEADA6_overlay.png
  CENT0008: F1=0.7750, Acc=0.9005
  Saved: auto_labeler/results/plots/CENT0008_overlay.png
  DD2DFAF4: F1=0.5429, Acc=0.7859
  Saved: auto_labeler/results/plots/DD2DFAF4_overlay.png
  F427536B: F1=0.9272, Acc=0.9626
  Saved: auto_labeler/results/plots/F427536B_overlay.png
  SUMM0127: F1=0.6509, Acc=0.8742
  Saved: auto_labeler/results/plots/SUMM0127_overlay.png

SUMMARY (8 studies)
  Accuracy:   0.8909 ± 0.0493
  F1 macro:   0.7193 ± 0.1167
  F1 blood : 0.9548 ± 0.0164
  F1 clot  : 0.3947 ± 0.2538
  F1 wall  : 0.8085 ± 0.1174
Results saved: auto_labe

In [ ]:
# View a sample overlay plot
from IPython.display import Image
import glob

plots = sorted(glob.glob("results/plots/*.png"))
if plots:
    print(f"Found {len(plots)} plots")
    display(Image(plots[0], width=1000))
else:
    print("No plots generated (test studies may not be in training_data/)")

## Download Trained Model

In [ ]:
# Checkpoint is already on Google Drive (saved in-place during training)
print("Checkpoint location: /content/drive/MyDrive/auto_labeler/checkpoints/best_model.pt")
!ls -la checkpoints/best_model.pt

## Predict Labels on New Data

Place unlabeled parquet files in `auto_labeler/predict_labels/`. Each file should have columns `timeInMS` and `magRLoadAdjusted`. The model will add a `predicted_label` column (0=blood, 1=clot, 2=wall) and overwrite the file in-place.

In [ ]:
%cd /content/drive/MyDrive/auto_labeler
import sys
sys.path.insert(0, "/content/drive/MyDrive")

import os
print(f"CWD: {os.getcwd()}")
print(f"Checkpoint exists: {os.path.exists('checkpoints/best_model.pt')}")
print(f"predict_labels/ exists: {os.path.exists('predict_labels')}")
if os.path.exists('predict_labels'):
    print(f"  Contents: {os.listdir('predict_labels')}")

import pandas as pd
import numpy as np
from pathlib import Path
from auto_labeler.predict import load_model, predict_file, postprocess_labels
from auto_labeler.dataset import build_multichannel
from auto_labeler import config as cfg
import torch

checkpoint = "checkpoints/best_model.pt"
predict_dir = Path("predict_labels")
predict_dir.mkdir(exist_ok=True)

# Find all parquet files to label
parquets = sorted(predict_dir.glob("*.parquet"))
print(f"\nFound {len(parquets)} files to label")
print(f"Post-processing: MIN_EVENT_DURATION_SEC = {cfg.MIN_EVENT_DURATION_SEC}")

if not parquets:
    print("No .parquet files found. Place files in /content/drive/MyDrive/auto_labeler/predict_labels/")
else:
    device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
    model, multichannel = load_model(checkpoint, device)
    print(f"Model loaded (multichannel={multichannel})")

    for pf in parquets:
        print(f"\n  Processing: {pf.name}")
        df = pd.read_parquet(pf)
        resistance = df["magRLoadAdjusted"].values.astype(np.float32)

        # Build features matching training
        if multichannel:
            features = build_multichannel(resistance)
        else:
            mean, std = resistance.mean(), resistance.std() + 1e-8
            features = ((resistance - mean) / std)[np.newaxis, :]

        # Run prediction + post-processing
        pred_labels = predict_file(model, features, device)
        pred_labels = postprocess_labels(pred_labels)

        # Add predicted_label column
        df["predicted_label"] = pred_labels

        # Save back
        df.to_parquet(pf, index=False)

        # Summary
        unique, counts = np.unique(pred_labels, return_counts=True)
        label_names = {0: "blood", 1: "clot", 2: "wall"}
        dist = ", ".join(f"{label_names[u]}: {c/len(pred_labels)*100:.1f}%" for u, c in zip(unique, counts))
        print(f"    Samples: {len(pred_labels):,} | {dist}")

    print(f"\nDone! {len(parquets)} files labeled and saved.")

/content/drive/MyDrive/auto_labeler
CWD: /content/drive/MyDrive/auto_labeler
Checkpoint exists: True
predict_labels/ exists: True
  Contents: ['33CFB812_labeled_segment.parquet']

Found 1 files to label
Model loaded (multichannel=False)

  Processing: 33CFB812_labeled_segment.parquet
    Samples: 351,627 | blood: 78.2%, clot: 2.1%, wall: 19.6%

Done! 1 files labeled and saved.


In [ ]:
# Generate interactive Plotly HTML files (downsampled for fast rendering)
!pip install plotly -q

import plotly.graph_objects as go
from plotly.subplots import make_subplots
import pandas as pd
import numpy as np
from pathlib import Path

predict_dir = Path("predict_labels")
output_dir = Path("results/plotly")
output_dir.mkdir(parents=True, exist_ok=True)

label_names = {0: "blood", 1: "clot", 2: "wall"}
label_colors = {0: "rgba(100,100,100,0.25)", 1: "rgba(255,0,0,0.45)", 2: "rgba(0,0,255,0.45)"}

# Max points to render (keeps HTML < 2MB, opens instantly)
MAX_POINTS = 8000


def min_max_downsample(x, y, max_points):
    """Downsample preserving min/max per bucket (keeps spikes visible)."""
    n = len(x)
    if n <= max_points:
        return x, y
    bucket_size = n // (max_points // 2)
    out_x, out_y = [], []
    for i in range(0, n, bucket_size):
        chunk = y[i : i + bucket_size]
        if len(chunk) == 0:
            continue
        i_min = i + np.argmin(chunk)
        i_max = i + np.argmax(chunk)
        for idx in sorted([i_min, i_max]):
            out_x.append(x[idx])
            out_y.append(y[idx])
    return np.array(out_x), np.array(out_y)


def get_label_segments(labels, time_sec):
    """Convert per-sample labels to (start_time, end_time, class) segments."""
    segments = []
    i = 0
    n = len(labels)
    while i < n:
        cls = labels[i]
        j = i + 1
        while j < n and labels[j] == cls:
            j += 1
        segments.append((time_sec[i], time_sec[min(j, n) - 1], int(cls)))
        i = j
    return segments


parquets = sorted(predict_dir.glob("*.parquet"))
print(f"Generating Plotly HTML for {len(parquets)} files...")

for pf in parquets:
    df = pd.read_parquet(pf)
    if "predicted_label" not in df.columns:
        print(f"  Skipping {pf.name} (no predicted_label column)")
        continue

    study_id = pf.stem.replace("_labeled_segment", "")
    resistance = df["magRLoadAdjusted"].values.astype(np.float32)
    pred_labels = df["predicted_label"].values
    has_gt = "label" in df.columns
    gt_labels = df["label"].values if has_gt else None

    # Time axis
    if "timeInMS" in df.columns:
        time_sec = df["timeInMS"].values / 1000.0
    else:
        time_sec = np.arange(len(resistance)) / 150.0

    # Downsample resistance for fast rendering
    t_ds, r_ds = min_max_downsample(time_sec, resistance, MAX_POINTS)

    # Get label segments
    pred_segments = get_label_segments(pred_labels, time_sec)
    gt_segments = get_label_segments(gt_labels, time_sec) if has_gt else []

    # Layout: resistance+predicted on top, GT below (if available)
    n_rows = 2 if has_gt else 1
    row_heights = [0.7, 0.3] if has_gt else [1.0]
    subtitles = [f"{study_id} — Resistance + Predicted"]
    if has_gt:
        subtitles.append("Ground Truth")

    fig = make_subplots(
        rows=n_rows, cols=1, shared_xaxes=True,
        row_heights=row_heights,
        vertical_spacing=0.08,
        subplot_titles=subtitles,
    )

    # Row 1: Resistance trace
    r_min, r_max = float(resistance.min()), float(resistance.max())
    r_pad = (r_max - r_min) * 0.02

    fig.add_trace(
        go.Scattergl(
            x=t_ds, y=r_ds, mode="lines",
            line=dict(color="black", width=0.8),
            name="Resistance",
            hovertemplate="t=%{x:.2f}s<br>R=%{y:.0f}Ω<extra></extra>",
        ),
        row=1, col=1,
    )

    # Add predicted label rectangles to row 1 using shapes with explicit axis refs
    for t0, t1, cls in pred_segments:
        if cls == 0:
            continue
        fig.add_shape(
            type="rect", x0=t0, x1=t1, y0=r_min - r_pad, y1=r_max + r_pad,
            xref="x", yref="y",
            fillcolor=label_colors[cls], line_width=0, layer="below",
        )

    # Row 2: GT rectangles (if available)
    if has_gt:
        # Add invisible trace so subplot renders with correct x range
        fig.add_trace(
            go.Scatter(x=[time_sec[0], time_sec[-1]], y=[0.5, 0.5],
                       mode="lines", line=dict(width=0, color="rgba(0,0,0,0)"),
                       showlegend=False, hoverinfo="skip"),
            row=2, col=1,
        )
        for t0, t1, cls in gt_segments:
            if cls == 0:
                continue
            fig.add_shape(
                type="rect", x0=t0, x1=t1, y0=0, y1=1,
                xref="x2", yref="y2",
                fillcolor=label_colors[cls], line_width=0, layer="below",
            )

    # Legend entries
    for cls in [1, 2]:
        fig.add_trace(go.Scatter(
            x=[None], y=[None], mode="markers",
            marker=dict(size=12, color=label_colors[cls]),
            name=label_names[cls], showlegend=True,
        ))

    fig.update_xaxes(title_text="Time (s)", row=n_rows, col=1)
    fig.update_yaxes(title_text="R (Ω)", range=[r_min - r_pad, r_max + r_pad], row=1, col=1)
    if has_gt:
        fig.update_yaxes(range=[0, 1], showticklabels=False, row=2, col=1)

    fig.update_layout(
        height=600 if has_gt else 450,
        width=1400,
        hovermode="x",
        showlegend=True,
        legend=dict(orientation="h", yanchor="bottom", y=1.02, xanchor="right", x=1),
    )

    out_path = output_dir / f"{study_id}_predicted.html"
    fig.write_html(str(out_path), include_plotlyjs="cdn")
    size_kb = out_path.stat().st_size / 1024
    gt_str = " (with GT)" if has_gt else ""
    print(f"  {study_id}: {size_kb:.0f} KB{gt_str} — {out_path}")

print(f"\nDone! HTML files in {output_dir}/")
print("Open in browser for instant zoom/pan/hover.")

## Layer Interpretability

Visualize what each encoder/decoder level learns. Shows activations aligned with the signal — zoom into boundaries to see how spatial precision recovers through the decoder.

In [ ]:
# Run layer interpretability on a test study
# Change TIME_START to focus on a different region (or None for auto = first clot)
%cd /content/drive/MyDrive/auto_labeler

STUDY_FILE = "test_data/8ECEADA6_labeled_segment.parquet"  # change as needed
TIME_START = None  # set to e.g. 500.0 to start chunk at 500 seconds

cmd = f'python interpret_layers.py "{STUDY_FILE}" --checkpoint checkpoints/best_model.pt'
if TIME_START is not None:
    cmd += f' --time_start {TIME_START}'
cmd += ' --output results/layer_activations.html'

import os
os.system(cmd)

# Display in notebook
from IPython.display import HTML, display
display(HTML('<a href="results/layer_activations.html" target="_blank">Open Layer Activations (full interactive)</a>'))
print("File saved: results/layer_activations.html")
print("Download and open in Chrome for full zoom/pan.")